# SQL Analytics

## Objective


Use the dimensional MySQL model developed in the previous stages to answer the project's business questions through analytical SQL queries.

This stage focuses on descriptive and exploratory analysis of well production, hydraulic fracturing, geological, operational, temporal, and spatial data.

The analysis follows the structure:

**Business Question → SQL Query → Result → Interpretation → Business Conclusion**

## Analytical Scope

The SQL analysis will address the business questions defined in the Business Understanding stage.

The analysis is organized into six analytical areas:

1. Production Performance
2. Temporal Analysis
3. Hydraulic Fracturing
4. Geological Analysis
5. Operational Analysis
6. Spatial Analysis

The analysis will use the dimensional model already implemented in MySQL.

No additional data loading, ETL, or modification of the fact tables will be performed in this stage.

## Dimensional Model

The SQL analysis will be performed using the following dimensional model:

### Fact Tables

- `fact_production`
- `fact_fracturing`

### Dimension Tables

- `dim_time`
- `dim_well`
- `dim_geology`
- `dim_field_area`
- `dim_location`
- `dim_company`

The fact tables contain the analytical measurements, while the dimension tables provide the temporal, well, geological, spatial, and company attributes required to analyze production and fracturing performance.

## Analytical Principles

The SQL analysis will follow these principles:

- Use only variables available in the dimensional model.
- Preserve the definitions established during the Business Understanding stage.
- Prefer transparent and reproducible SQL queries.
- Distinguish descriptive patterns from causal relationships.
- Report limitations when data coverage or completeness affects interpretation.
- Avoid unsupported assumptions or artificial imputation.
- Validate analytical results before drawing business conclusions.

The purpose of this stage is to identify patterns and associations that can later be investigated through Python-based statistical analysis.

## MySQL Connection

In [1]:
import pandas as pd
import mysql.connector

In [2]:
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="123456789"
)

cursor = conn.cursor(buffered=True)
cursor.execute("USE oil_gas_dw")

print("Database selected.")

Database selected.


## BQ-01 — How does well production evolve over time?

### Business Question

How does well production evolve over time?

### Analytical Approach

Production will be aggregated monthly to evaluate the temporal evolution of oil and gas production.

The production fact table will be joined with the time dimension through `time_key`.

### SQL Query

In [3]:
query_bq01 = """
SELECT
    t.year,
    t.month,
    SUM(fp.prod_pet) AS total_oil_production,
    SUM(fp.prod_gas) AS total_gas_production
FROM fact_production AS fp
JOIN dim_time AS t
    ON fp.time_key = t.time_key
GROUP BY
    t.year,
    t.month
ORDER BY
    t.year,
    t.month;
"""

### Result

In [4]:
df_bq01 = pd.read_sql(query_bq01, conn)

df_bq01.head()

C:\Users\Admin\AppData\Local\Temp\ipykernel_10924\2014480578.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_bq01 = pd.read_sql(query_bq01, conn)


,year,month,total_oil_production,total_gas_production
0,2006,1,1070.01,32508.00
1,2006,2,994.52,30792.89
2,2006,3,1136.85,33042.57
3,2006,4,1113.70,31362.11
4,2006,5,1505.22,39218.74


Control 1: Month of maximum and minimum production

In [5]:
query_control_01 = """
SELECT
    min(t.full_date),
    max(t.full_date)
FROM dim_time AS t
JOIN fact_production AS fp
    ON fp.time_key = t.time_key
"""

df_control_01 = pd.read_sql(query_control_01, conn)
df_control_01

C:\Users\Admin\AppData\Local\Temp\ipykernel_10924\2134265284.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_control_01 = pd.read_sql(query_control_01, conn)


,min(t.full_date),max(t.full_date)
0,2006-01-31,2026-06-30


Control 2: Months with no production records

In [6]:
query_control_02 = """
SELECT
    cal.year,
    cal.month
FROM (
    SELECT DISTINCT
        year,
        month
    FROM dim_time
    WHERE full_date <= (
        SELECT MAX(t2.full_date)
        FROM dim_time AS t2
        JOIN fact_production AS fp2
            ON t2.time_key = fp2.time_key
    )
) AS cal
LEFT JOIN (
    SELECT DISTINCT
        t.year,
        t.month
    FROM fact_production AS fp
    JOIN dim_time AS t
        ON fp.time_key = t.time_key
) AS prod
    ON cal.year = prod.year
    AND cal.month = prod.month
WHERE prod.year IS NULL
ORDER BY
    cal.year,
    cal.month;
"""

df_control_02 = pd.read_sql(query_control_02, conn)
df_control_02

C:\Users\Admin\AppData\Local\Temp\ipykernel_10924\2095310745.py:33: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_control_02 = pd.read_sql(query_control_02, conn)


,year,month


### Interpretation

The monthly production of oil and gas can be analyzed from January 2006 to June 2026.

Control 01 confirmed the actual production period, while Control 02 confirmed that there are no complete months without production records within this period.

However, these results are descriptive and do not allow us to determine why production increases or decreases. The BQ-01 analysis describes the overall monthly production trend across all wells but does not identify the factors associated with changes in production.

### Business Conclusion

Recorded production shows a continuous monthly time series from January 2006 to June 2026, with no complete months missing from the production records within this period.

## BQ-02 Which wells achieve the highest cumulative production?

### Business Question

Which wells achieve the highest cumulative production?

### Analytical Approach

Calculate cumulative oil and gas production for each well by aggregating all available production records across the well's production history.

Oil and gas production will be analyzed separately because they are measured in different units.

The wells will then be ranked by cumulative production to identify those with the highest recorded oil and gas output.

### SQL

In [14]:
query_bq02_oil = """
SELECT 
    well_key,
    SUM(prod_pet) AS cumulative_oil_production,
    SUM(prod_gas) AS cumulative_gas_production
FROM fact_production
GROUP BY well_key
HAVING cumulative_oil_production != 0 
    OR cumulative_gas_production != 0
ORDER BY cumulative_oil_production DESC
LIMIT 10;
"""

### Result

In [15]:
df_bq02_oil = pd.read_sql(query_bq02_oil, conn)
df_bq02_oil

C:\Users\Admin\AppData\Local\Temp\ipykernel_10924\1618429688.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_bq02_oil = pd.read_sql(query_bq02_oil, conn)


,well_key,cumulative_oil_production,cumulative_gas_production
0,8688,337398.08,29323.18
1,7123,243289.05,39119.97
2,9840,234654.13,107702.32
3,6462,229963.66,24752.81
4,6931,219702.87,18918.53
5,8562,213812.54,10623.53
6,8629,198351.26,28169.87
7,8645,194244.57,26474.44
8,7120,193246.17,31979.36
9,7795,184403.40,18092.88


In [17]:
query_bq02_gas = """
SELECT 
    well_key,
    SUM(prod_pet) AS cumulative_oil_production,
    SUM(prod_gas) AS cumulative_gas_production
FROM fact_production
GROUP BY well_key
HAVING cumulative_oil_production != 0 
    OR cumulative_gas_production != 0
ORDER BY cumulative_gas_production DESC
LIMIT 10;
"""

In [18]:
df_bq02_gas = pd.read_sql(query_bq02_gas, conn)
df_bq02_gas

C:\Users\Admin\AppData\Local\Temp\ipykernel_10924\1302759985.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_bq02_gas = pd.read_sql(query_bq02_gas, conn)


,well_key,cumulative_oil_production,cumulative_gas_production
0,9943,705.70,743837.25
1,9948,320.08,680255.15
2,9915,3633.88,659744.37
3,9885,6391.28,651211.48
4,9924,1404.09,638858.00
5,8930,14042.73,633618.58
6,9976,107.93,589759.14
7,9139,8445.96,558050.32
8,9957,121.35,538323.81
9,8899,21682.42,531463.76


### Interpretation

The results show that the wells with the highest cumulative oil production are not necessarily the same wells that achieve the highest cumulative gas production.

The oil ranking is led by well 8688, with 337,398.08 units of cumulative oil production, while the gas ranking is led by well 9943, with 743,837.25 units of cumulative gas production.

The gas ranking also includes several wells with relatively low cumulative oil production. For example, well 9943 accumulated 743,837.25 units of gas but only 705.70 units of oil.

These results indicate that well production performance varies depending on the hydrocarbon considered. However, the rankings are descriptive and do not establish the geological or operational factors associated with these differences.

### Business Conclusion

The analysis shows that the wells with the highest cumulative oil production are not necessarily the same wells with the highest cumulative gas production. This indicates that well performance differs depending on the hydrocarbon considered. The results provide an initial identification of high-performing wells and establish a basis for further analysis of the geological and operational factors associated with these production patterns.

## BQ-03 Which wells exhibit the fastest production decline over time?

### Business Question

Which wells exhibit the fastest production decline over time?

### Analytical Approach

Analyze the production history of each well over time to identify wells that experience the fastest decline in oil and gas production.

For each well, production records will be ordered chronologically and the change between the beginning and the end of its production history will be evaluated.

Oil and gas production will be analyzed separately because they are measured in different units.

The decline will be evaluated relative to the duration of the available production history so that wells with different observation periods can be compared more consistently.